In [ ]:

QM9_XYZ_DIR = '/kaggle/input/datasets/zaharch/quantum-machine-9-aka-qm9'

# set to true for a fast 5-epoch sanity check on 5000 molecules.
# set to false for the full benchmark (50 epochs, all molecules).
QUICK_RUN = False

# number of training epochs per target.
EPOCHS = 5 if QUICK_RUN else 50

# maximum number of xyz files to load. none = load everything in the folder.
MAX_MOLECULES = 5000 if QUICK_RUN else None

# batch size. 32 fits comfortably on a t4. reduce to 16 if oom.
BATCH_SIZE = 32

# names and xyz-file property-line positions for the five targets.
# position refers to the 0-based column index in the properties line
# of the qm9 xyz file, which has the structure:
#   gdb [idx] [A] [B] [C] [mu] [alpha] [homo] [lumo] [gap] [r2] ...
TARGETS = [
    {'name': 'mu    (dipole moment, debye)',        'xyz_pos': 5,  'col': 0},
    {'name': 'alpha (polarizability, bohr3)',        'xyz_pos': 6,  'col': 1},
    {'name': 'homo  (homo energy, hartree)',         'xyz_pos': 7,  'col': 2},
    {'name': 'lumo  (lumo energy, hartree)',         'xyz_pos': 8,  'col': 3},
    {'name': 'gap   (homo-lumo gap, hartree)',       'xyz_pos': 9,  'col': 4},
]

# split ratios.
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
# test is the remainder = 0.1

# random seed.
SEED = 42

# transformer hyperparameters.
# d_model and n_layers are reduced from the preprint defaults (768, 6) so
# the benchmark finishes in a reasonable time on kaggle without a p100.
D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 8
D_K        = 32
D_V        = 32
D_FF       = 1024
VOCAB_SIZE = 83
MAXLEN     = 501

# gcn hyperparameters.
GCN_LAYERS   = 4
GCN_EMB_DIM  = 256
GCN_FEAT_DIM = 128
GCN_DROP     = 0.2

# combined feature dimension entering the predictor mlp.
COMBINED_DIM = D_MODEL + GCN_FEAT_DIM

# optimizer settings.
LEARNING_RATE = 2e-4
WEIGHT_DECAY  = 5e-6

# ============================================================
# section 1: dependency installation
# ============================================================

# ============================================================
# section 1: dependency installation
# ============================================================

import subprocess
import sys

def install(package):
    # install a package silently.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('installing dependencies... (using pre-built wheels to prevent hanging)')
install('torch_geometric')

# --- FIX: Use PyG wheels for scatter and sparse ---
import torch
torch_version = torch.__version__.split('+')[0]
cuda_str = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_str}.html'

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch_scatter', 'torch_sparse', '-f', wheel_url])
except Exception as e:
    print("Warning: PyG wheels failed, falling back to native.")

# --------------------------------------------------

install('rdkit')
install('nltk')
install('six')

import nltk
nltk.download('punkt', quiet=True)

print('all dependencies installed.')

# ============================================================
# section 2: imports
# ============================================================

import os
import sys
import copy
import math
import time
import glob
import contextlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from rdkit import Chem
from rdkit.Chem.rdchem import BondType as BT
from rdkit import RDLogger

import six

# suppress rdkit log noise.
RDLogger.DisableLog('rdApp.*')

from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import add_self_loops
from torch_scatter import scatter_add

# ============================================================
# section 3: zinc cfg grammar and token sequence tools
# ============================================================
# this section is fully self-contained. it reproduces the zinc grammar
# from preprocess/grammar.py and the encoding logic so the script
# runs on kaggle without the repository's preprocess/ directory.

GRAM_STRING = """smiles -> chain
atom -> bracket_atom
atom -> aliphatic_organic
atom -> aromatic_organic
aliphatic_organic -> 'B'
aliphatic_organic -> 'C'
aliphatic_organic -> 'N'
aliphatic_organic -> 'O'
aliphatic_organic -> 'S'
aliphatic_organic -> 'P'
aliphatic_organic -> 'F'
aliphatic_organic -> 'I'
aliphatic_organic -> 'Cl'
aliphatic_organic -> 'Br'
aromatic_organic -> 'c'
aromatic_organic -> 'n'
aromatic_organic -> 'o'
aromatic_organic -> 's'
bracket_atom -> '[' BAI ']'
BAI -> isotope symbol BAC
BAI -> symbol BAC
BAI -> isotope symbol
BAI -> symbol
BAC -> chiral BAH
BAC -> BAH
BAC -> chiral
BAH -> hcount BACH
BAH -> BACH
BAH -> hcount
BACH -> charge class
BACH -> charge
BACH -> class
symbol -> aliphatic_organic
symbol -> aromatic_organic
isotope -> DIGIT
isotope -> DIGIT DIGIT
isotope -> DIGIT DIGIT DIGIT
DIGIT -> '1'
DIGIT -> '2'
DIGIT -> '3'
DIGIT -> '4'
DIGIT -> '5'
DIGIT -> '6'
DIGIT -> '7'
DIGIT -> '8'
chiral -> '@'
chiral -> '@@'
hcount -> 'H'
hcount -> 'H' DIGIT
charge -> '-'
charge -> '-' DIGIT
charge -> '-' DIGIT DIGIT
charge -> '+'
charge -> '+' DIGIT
charge -> '+' DIGIT DIGIT
bond -> '-'
bond -> '='
bond -> '#'
bond -> '/'
bond -> '\\\\'
bond -> '.'
ringbond -> DIGIT
ringbond -> bond DIGIT
branched_atom -> atom
branched_atom -> atom RB
branched_atom -> atom BB
branched_atom -> atom RB BB
RB -> RB ringbond
RB -> ringbond
BB -> BB branch
BB -> branch
branch -> '(' chain ')'
branch -> '(' bond chain ')'
chain -> branched_atom
chain -> chain branched_atom
chain -> chain bond branched_atom
symbol -> element_symbols
aromatic_organic -> 'p'
element_symbols -> 'H'
class -> DIGIT
Nothing -> None"""

# parse the grammar into an nltk cfg object.
# this is done once at import time, not per molecule.
GCFG = nltk.CFG.fromstring(GRAM_STRING)

# build auxiliary data structures required by the grammar encoder.
# these mirror grammar.py in the repository.
_all_lhs = [a.lhs().symbol() for a in GCFG.productions()]
_lhs_list = []
for a in _all_lhs:
    if a not in _lhs_list:
        _lhs_list.append(a)

# pre-build the nltk chart parser once. creating a ChartParser object
# is expensive; reusing a single instance avoids that overhead per molecule.
_PARSER = nltk.ChartParser(GCFG)

# pre-build a lookup: (lhs_symbol, rhs_tuple) -> 1-based rule index.
# this makes the tree traversal O(1) per node instead of O(D) per node.
_RULE_LOOKUP = {}
for idx, prod in enumerate(GCFG.productions()):
    lhs = prod.lhs().symbol()
    rhs = tuple(
        s.symbol() if not isinstance(s, str) else s
        for s in prod.rhs()
    )
    _RULE_LOOKUP[(lhs, rhs)] = idx + 1  # 1-based token integer

import re as _re
_SMILES_TOKENIZER = _re.compile(
    r'(\[[^\]]+\]|Br|Cl|@@|[BCNOPSFIHhbcnosp=#@+\-\/\\\.1-9\(\)])'
)


def _tokenize(smiles):
    # split a smiles string into individual symbols.
    # handles multi-character tokens like 'Cl', 'Br', '[nH]', '@@'.
    return _SMILES_TOKENIZER.findall(smiles)


def encode_smiles(smiles):
    # encode a smiles string into a list of 1-based grammar rule integers.
    # raises ValueError if the smiles cannot be parsed by the zinc cfg.
    #
    # the cfg parse tree is traversed in pre-order. each internal node
    # (non-leaf) corresponds to one rule application. the rule's 1-based
    # index in GCFG.productions() is appended to the output list.

    tokens = _tokenize(smiles)
    if not tokens:
        raise ValueError(f'smiles tokenizer returned empty list: {smiles}')

    trees = list(_PARSER.parse(tokens))
    if not trees:
        raise ValueError(f'nltk cfg parse failed: {smiles}')

    tree = trees[0]

    rule_indices = []
    for subtree in tree.subtrees():
        if subtree.height() > 1:
            lhs = subtree.label()
            rhs = tuple(
                child.label() if isinstance(child, nltk.Tree) else child
                for child in subtree
            )
            rule_idx = _RULE_LOOKUP.get((lhs, rhs))
            if rule_idx is not None:
                rule_indices.append(rule_idx)

    return rule_indices


# valid grammar token integers are 1 to 80.
# token 0 = [PAD], token 82 = [GLO].
_GRAMMAR_TOKEN_SET = set(range(1, 81))


def construct_token_sequence(rule_index_list, max_len=500):
    # convert a list of grammar rule integers to the fixed-length
    # token sequence used by the transformer encoder.
    #
    # returns:
    #   tokens_idx : numpy int64 array of shape (501,)
    #                values: 82 at position 0 ([GLO]), 1-80 for grammar tokens,
    #                0 for [PAD] positions.
    #   atom_mask  : numpy int64 array of shape (501,)
    #                1 at positions with real tokens ([GLO] or grammar), 0 for [PAD].

    if len(rule_index_list) > max_len:
        rule_index_list = rule_index_list[:max_len]

    padding_len = max_len - len(rule_index_list)
    tokens_idx  = np.zeros(max_len + 1, dtype=np.int64)
    atom_mask   = np.zeros(max_len + 1, dtype=np.int64)

    # position 0 is always [GLO] (index 82).
    tokens_idx[0] = 82
    atom_mask[0]  = 1

    for i, token in enumerate(rule_index_list):
        pos = i + 1
        if token in _GRAMMAR_TOKEN_SET:
            tokens_idx[pos] = token
            atom_mask[pos]  = 1
        # tokens outside the grammar set are left as 0 ([PAD]).

    # padding positions remain 0 with atom_mask 0.
    return tokens_idx, atom_mask


# ============================================================
# section 4: electron-relevant token set
# ============================================================
# reproduced from preprocess/property_tags.py.
# these are the 1-based token integers (= 0-based production index + 1)
# corresponding to:
#   aliphatic heteroatoms: N=7, O=8, S=9
#   aromatic atoms:        c=15, n=16, o=17, s=18, p=78
#   pi bonds:              '='=57, '#'=58

ELECTRON_RELEVANT_TOKEN_SET = {7, 8, 9, 15, 16, 17, 18, 57, 58, 78}

# ============================================================
# section 5: model architecture
# ============================================================
# reproduced from model/my_nn.py (property-aware version),
# model/gcn_finetune.py, and model/my_fusion_model.py.
# all classes are inlined here so the script is self-contained on kaggle.

def get_attn_pad_mask(seq_q):
    # build the padding attention mask.
    # returns shape [batch, seq_len, seq_len].
    # true wherever seq_q == 0 ([PAD]), causing attention to ignore those positions.
    batch_size, seq_len = seq_q.size()
    pad_attn_mask = seq_q.data.eq(0).unsqueeze(1)
    return pad_attn_mask.expand(batch_size, seq_len, seq_len)


class Embedding(nn.Module):
    # property-aware embedding layer.
    # implements the modified formula:
    #   h_0 = LayerNorm( E_tok(x) + E_pos(p) + electron_mask(x) * beta )
    # where beta is a single learnable scalar initialized to 0.1.
    # see PROPERTY_AWARE_GRAMMAR_CHANGES.md section 4 for full derivation.

    def __init__(self, vocab_size, d_model, maxlen, electron_token_set):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(maxlen, d_model)
        self.norm = nn.LayerNorm(d_model)

        # learnable scalar bias for electron-relevant positions.
        # initialized to 0.1. updated by the optimizer during training.
        # if the property has no electron-relevance signal, gradient descent
        # will drive this value toward zero. tracking it per epoch shows
        # whether the signal is being used.
        self.electron_bias = nn.Parameter(torch.tensor(0.1))

        # fixed lookup buffer of electron-relevant token integers.
        # stored as a registered buffer so it moves to the correct device
        # with model.to(device) without any manual .to() calls.
        self.register_buffer(
            'electron_token_indices',
            torch.tensor(sorted(electron_token_set), dtype=torch.long)
        )

    def forward(self, x):
        # x: [batch, seq_len] integer token indices.
        seq_len = x.size(1)

        # build positional index sequence [0, 1, ..., seq_len-1] over the batch.
        pos = torch.arange(seq_len, dtype=torch.long).unsqueeze(0).expand_as(x).to(x.device)

        tok_embedding = self.tok_embed(x)    # [batch, seq_len, d_model]
        pos_embedding = self.pos_embed(pos)  # [batch, seq_len, d_model]

        # construct the binary electron-relevance mask.
        # x.unsqueeze(-1) shape: [batch, seq_len, 1]
        # electron_token_indices shape: [num_electron_tokens]
        # broadcast comparison: [batch, seq_len, num_electron_tokens]
        # .any(dim=-1): [batch, seq_len] boolean
        # .unsqueeze(-1).float(): [batch, seq_len, 1] binary float
        electron_mask = (x.unsqueeze(-1) == self.electron_token_indices).any(dim=-1)
        electron_mask = electron_mask.unsqueeze(-1).float()

        # multiply: electron_mask [batch, seq_len, 1] * scalar -> [batch, seq_len, 1]
        # this broadcasts to [batch, seq_len, d_model] when added to the embeddings.
        # non-electron positions contribute 0.0 * electron_bias = 0.
        electron_bump = electron_mask * self.electron_bias

        return self.norm(tok_embedding + pos_embedding + electron_bump)


class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q, K, V, attn_mask):
        # standard scaled dot-product attention.
        # scores: [batch, n_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)
        scores.masked_fill_(attn_mask, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)
        self.linear = nn.Linear(n_heads * d_v, d_model)
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, Q, K, V, attn_mask):
        residual, batch_size = Q, Q.size(0)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1, 2)
        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        context = ScaledDotProductAttention(self.d_k)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v)
        return self.layernorm(self.linear(context) + residual)


class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.ReLU(),
            nn.Linear(d_ff, d_model, bias=False),
        )
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, inputs):
        return self.layernorm(self.fc(inputs) + inputs)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads, d_ff):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, d_k, d_v, n_heads)
        self.pos_ffn = PoswiseFeedForwardNet(d_model, d_ff)

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)
        return self.pos_ffn(enc_outputs)


class TransformerEncoder(nn.Module):
    # property-aware transformer encoder (BERT_atom_embedding_generator equivalent).
    # input:  token_ids of shape [batch, 501]
    # output: h_global of shape [batch, d_model] — the [GLO] token representation.

    def __init__(self, d_model, n_layers, vocab_size, maxlen, d_k, d_v, n_heads, d_ff,
                 electron_token_set):
        super().__init__()
        # the Embedding module carries the electron_bias parameter.
        self.embedding = Embedding(vocab_size, d_model, maxlen, electron_token_set)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, d_k, d_v, n_heads, d_ff) for _ in range(n_layers)
        ])

    def forward(self, input_ids):
        # step 1: compute h_0 with property-aware embedding.
        output = self.embedding(input_ids)               # [batch, 501, d_model]
        # step 2: build padding mask so attention ignores [PAD] positions.
        attn_mask = get_attn_pad_mask(input_ids)        # [batch, 501, 501]
        # step 3: run encoder stack.
        for layer in self.layers:
            output = layer(output, attn_mask)
        # step 4: return [GLO] token at position 0 as the global representation.
        return output[:, 0, :]                           # [batch, d_model]


# gcn branch constants.
_NUM_ATOM_TYPE      = 119
_NUM_CHIRALITY_TAG  = 3
_NUM_BOND_TYPE      = 5
_NUM_BOND_DIRECTION = 3


def _gcn_norm(edge_index, num_nodes=None):
    # symmetric normalized adjacency: D^{-1/2} A D^{-1/2}.
    # used inside each GCN layer to prevent gradient explosion.
    from torch_geometric.utils.num_nodes import maybe_num_nodes
    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    edge_weight = torch.ones(edge_index.size(1), device=edge_index.device)
    row, col = edge_index[0], edge_index[1]
    deg = scatter_add(edge_weight, col, dim=0, dim_size=num_nodes)
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]


from torch_geometric.nn import MessagePassing
import torch_sparse


class GCNConvLayer(MessagePassing):
    # one graph convolution layer.
    # matches GCNConv in model/gcn_finetune.py exactly.

    def __init__(self, emb_dim, aggr='add'):
        super().__init__()
        self.emb_dim = emb_dim
        self.aggr = aggr
        self.weight = nn.Parameter(torch.Tensor(emb_dim, emb_dim))
        self.bias   = nn.Parameter(torch.Tensor(emb_dim))
        stdv = math.sqrt(6.0 / (emb_dim + emb_dim))
        self.weight.data.uniform_(-stdv, stdv)
        self.bias.data.fill_(0)
        self.edge_embedding1 = nn.Embedding(_NUM_BOND_TYPE, 1)
        self.edge_embedding2 = nn.Embedding(_NUM_BOND_DIRECTION, 1)
        nn.init.xavier_uniform_(self.edge_embedding1.weight.data)
        nn.init.xavier_uniform_(self.edge_embedding2.weight.data)

    def forward(self, x, edge_index, edge_attr):
        # add self-loops to preserve the atom's own features during aggregation.
        edge_index = add_self_loops(edge_index, num_nodes=x.size(0))[0]
        self_loop_attr = torch.zeros(x.size(0), 2, device=edge_attr.device, dtype=edge_attr.dtype)
        self_loop_attr[:, 0] = 4  # bond type index 4 = self-loop (no bond)
        edge_attr = torch.cat((edge_attr, self_loop_attr), dim=0)
        edge_embeddings = self.edge_embedding1(edge_attr[:, 0]) + self.edge_embedding2(edge_attr[:, 1])
        edge_index, _ = _gcn_norm(edge_index)
        x = x @ self.weight
        out = self.propagate(edge_index, x=x, edge_attr=edge_embeddings, size=None)
        return out + self.bias

    def message(self, x_j, edge_attr):
        return x_j if edge_attr is None else edge_attr + x_j

    def message_and_aggregate(self, adj_t, x):
        return torch_sparse.matmul(adj_t, x, reduce=self.aggr)


class GCN(nn.Module):
    # multi-layer graph convolutional network.
    # produces a fixed-size graph-level embedding via mean pooling.
    # matches model/gcn_finetune.py.

    def __init__(self, num_layer=4, emb_dim=256, feat_dim=128, drop_ratio=0.2):
        super().__init__()
        if num_layer < 2:
            raise ValueError('gcn requires at least 2 layers.')
        self.num_layer  = num_layer
        self.drop_ratio = drop_ratio
        self.x_embedding1 = nn.Embedding(_NUM_ATOM_TYPE,     emb_dim)
        self.x_embedding2 = nn.Embedding(_NUM_CHIRALITY_TAG, emb_dim)
        nn.init.xavier_uniform_(self.x_embedding1.weight.data)
        nn.init.xavier_uniform_(self.x_embedding2.weight.data)
        self.phys_lin    = nn.Linear(4, emb_dim)
        self.gnns        = nn.ModuleList([GCNConvLayer(emb_dim) for _ in range(num_layer)])
        self.batch_norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(num_layer)])
        self.pool        = global_mean_pool
        self.feat_lin    = nn.Linear(emb_dim, feat_dim)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.x_embedding1(x[:, 0].long()) + self.x_embedding2(x[:, 1].long())
        h = h + self.phys_lin(x[:, 2:6])
        for i in range(self.num_layer):
            h = self.gnns[i](h, edge_index, edge_attr)
            h = self.batch_norms[i](h)
            if i == self.num_layer - 1:
                h = F.dropout(h, self.drop_ratio, training=self.training)
            else:
                h = F.dropout(F.relu(h), self.drop_ratio, training=self.training)
        h = self.pool(h, batch)
        return self.feat_lin(h)


class CombinedModel(nn.Module):
    # fuses transformer and gcn outputs by concatenation.
    # output shape: [batch, d_model + feat_dim]

    def __init__(self, transformer, gcn):
        super().__init__()
        self.transformer = transformer
        self.gcn = gcn

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        h_transformer = self.transformer(token_idx)
        h_gcn = self.gcn(x, edge_index, edge_attr, batch)
        return torch.cat([h_transformer, h_gcn], dim=1)


class Predictor(nn.Module):
    # two-layer mlp predicting a single regression target.

    def __init__(self, in_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim * 2),
            nn.Dropout(p=dropout),
            nn.ReLU(),
            nn.Linear(in_dim * 2, 1),
        )

    def forward(self, x):
        return self.net(x)


class FullModel(nn.Module):
    def __init__(self, combined, predictor):
        super().__init__()
        self.combined  = combined
        self.predictor = predictor

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        return self.predictor(self.combined(token_idx, x, edge_index, edge_attr, batch))


# ============================================================
# section 6: qm9 xyz file parsing
# ============================================================

# rdkit atom and bond feature lists.
# these match the existing dataloaders in the repository exactly.
ELECTRONEGATIVITY = {
    1: 2.20, 2: 0.00, 3: 0.98, 4: 1.57, 5: 2.04, 6: 2.55, 7: 3.04, 8: 3.44, 9: 3.98, 10: 0.00,
    11: 0.93, 12: 1.31, 13: 1.61, 14: 1.90, 15: 2.19, 16: 2.58, 17: 3.16, 18: 0.00,
    19: 0.82, 20: 1.00, 26: 1.83, 29: 1.90, 30: 1.65, 35: 2.96, 53: 2.66
}

VALENCE_ELECTRONS = {
    1: 1, 2: 2, 3: 1, 4: 2, 5: 3, 6: 4, 7: 5, 8: 6, 9: 7, 10: 8,
    11: 1, 12: 2, 13: 3, 14: 4, 15: 5, 16: 6, 17: 7, 18: 8,
    19: 1, 20: 2, 26: 2, 29: 1, 30: 2, 35: 7, 53: 7
}

BOND_ORDER = {
    BT.SINGLE: 1,
    BT.DOUBLE: 2,
    BT.TRIPLE: 3,
    BT.AROMATIC: 4,
}

_ATOM_LIST = list(range(1, 119))
_CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
_BOND_LIST = [BT.SINGLE, BT.DOUBLE, BT.TRIPLE, BT.AROMATIC]
_BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]


def _parse_float(s):
    # parse a float that may use mathematica scientific notation.
    # the standard qm9 xyz files from figshare use '1.23*^-4' instead of '1.23e-4'.
    # this handles both formats.
    return float(s.replace('*^', 'e'))


def parse_xyz_file(filepath):
    # parse one qm9 xyz file.
    # returns (smiles, targets_dict) or raises an exception on malformed files.
    #
    # qm9 xyz file structure:
    #   line 0:    number of atoms (na)
    #   line 1:    "gdb [idx] [A] [B] [C] [mu] [alpha] [homo] [lumo] [gap] [r2] ..."
    #   lines 2 to na+1: "element x y z mulliken_charge"
    #   line na+2: "SMILES_gdb\tSMILES_b3lyp"
    #   line na+3: "InChI..."

    with open(filepath, 'r') as f:
        lines = f.read().splitlines()

    na = int(lines[0].strip())

    # parse the 17 scalar properties from line 1.
    # the format is: gdb [idx] [A] [B] [C] [mu] [alpha] [homo] [lumo] [gap] ...
    props = lines[1].split()

    targets = {
        'mu':    _parse_float(props[5]),   # dipole moment (debye)
        'alpha': _parse_float(props[6]),   # isotropic polarizability (bohr^3)
        'homo':  _parse_float(props[7]),   # homo orbital energy (hartree)
        'lumo':  _parse_float(props[8]),   # lumo orbital energy (hartree)
        'gap':   _parse_float(props[9]),   # homo-lumo gap (hartree)
    }

    # atomic frequencies are at na+2, SMILES line is at na+3.
    smiles_line = lines[na + 3]

    # the smiles line contains two smiles separated by a tab.
    # the first is from gdb-17, the second is from the b3lyp optimization.
    # we use the first (gdb-17) smiles as it is canonical and shorter.
    smiles = smiles_line.split('\t')[0].strip()

    return smiles, targets


def smiles_to_graph_features(smiles):
    # build rdkit-based graph features for the gcn branch.
    # matches data_process_FreeSolv.py in the repository exactly:
    # explicit hydrogens are added as nodes (Chem.AddHs).
    #
    # returns (x, edge_index, edge_attr) or None if rdkit fails.

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    N = mol.GetNumAtoms()

    type_idx      = []
    chirality_idx = []
    atomic_nums   = []
    electroneg    = []
    valence_e     = []

    for atom in mol.GetAtoms():
        try:
            type_idx.append(_ATOM_LIST.index(atom.GetAtomicNum()))
        except ValueError:
            type_idx.append(0)
        try:
            chirality_idx.append(_CHIRALITY_LIST.index(atom.GetChiralTag()))
        except ValueError:
            chirality_idx.append(0)

        z = atom.GetAtomicNum()
        atomic_nums.append(float(z))
        electroneg.append(ELECTRONEGATIVITY.get(z, 0.0))
        valence_e.append(float(VALENCE_ELECTRONS.get(z, 0)))

    x1 = torch.tensor(type_idx,      dtype=torch.long).view(-1, 1)
    x2 = torch.tensor(chirality_idx, dtype=torch.long).view(-1, 1)
    x3 = torch.tensor(atomic_nums,   dtype=torch.float).view(-1, 1)
    x4 = torch.tensor(electroneg,    dtype=torch.float).view(-1, 1)
    x5 = torch.tensor(valence_e,     dtype=torch.float).view(-1, 1)

    bond_order_per_atom = [0.0] * N
    for bond in mol.GetBonds():
        order = float(BOND_ORDER.get(bond.GetBondType(), 1))
        i_start = bond.GetBeginAtomIdx()
        i_end   = bond.GetEndAtomIdx()
        bond_order_per_atom[i_start] = max(bond_order_per_atom[i_start], order)
        bond_order_per_atom[i_end]   = max(bond_order_per_atom[i_end],   order)

    x6 = torch.tensor(bond_order_per_atom, dtype=torch.float).view(-1, 1)

    x_int   = torch.cat([x1, x2], dim=-1)
    x_float = torch.cat([x3, x4, x5, x6], dim=-1)
    x = torch.cat([x_int.float(), x_float], dim=-1)

    row, col, edge_feat = [], [], []
    for bond in mol.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [start, end]
        col += [end, start]
        try:
            bt = _BOND_LIST.index(bond.GetBondType())
        except ValueError:
            bt = 0
        try:
            bd = _BONDDIR_LIST.index(bond.GetBondDir())
        except ValueError:
            bd = 0
        edge_feat += [[bt, bd], [bt, bd]]

    if not row:
        # single-atom molecule. add a self-loop so edge_index is non-empty.
        row, col   = [0], [0]
        edge_feat  = [[4, 0]]

    edge_index = torch.tensor([row, col], dtype=torch.long)
    edge_attr  = torch.tensor(edge_feat,  dtype=torch.long)
    return x, edge_index, edge_attr


# ============================================================
# section 7: one-time preprocessing of all molecules
# ============================================================

@contextlib.contextmanager
def _suppress_stderr():
    # redirect stderr to /dev/null to silence rdkit and nltk noise.
    with open(os.devnull, 'w') as devnull:
        old = sys.stderr
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stderr = old


def preprocess_all_molecules(xyz_dir, max_molecules=None):
    # scan the xyz directory, parse all files, encode through the cfg,
    # and build rdkit graph features.
    #
    # this function runs exactly once. all five target training loops
    # share the output without re-parsing anything.
    #
    # returns:
    #   cached_samples : list of dicts, each with keys:
    #     'tokens_idx' : numpy int64 array (501,)
    #     'atom_mask'  : numpy int64 array (501,)
    #     'targets'    : numpy float32 array (5,) — [mu, alpha, homo, lumo, gap]
    #     'x'          : torch.long tensor [n_atoms, 2]
    #     'edge_index' : torch.long tensor [2, 2*n_bonds]
    #     'edge_attr'  : torch.long tensor [2*n_bonds, 2]

    # find all xyz files and sort by name so the order is reproducible.
    pattern = os.path.join(xyz_dir, '*.xyz')
    xyz_files = sorted(glob.glob(pattern))

    if not xyz_files:
        raise FileNotFoundError(
            f'no .xyz files found in {xyz_dir}. '
            f'check the QM9_XYZ_DIR setting at the top of the script.'
        )

    if max_molecules is not None:
        xyz_files = xyz_files[:max_molecules]

    print(f'found {len(xyz_files)} xyz files. starting preprocessing...')
    print('this runs once. subsequent target training loops reuse the cache.')
    print()

    cached_samples = []
    n_failed_xyz    = 0
    n_failed_parse  = 0
    n_failed_rdkit  = 0
    t_start         = time.time()

    for i, filepath in enumerate(xyz_files):

        if i > 0 and i % 5000 == 0:
            elapsed   = time.time() - t_start
            rate      = i / elapsed
            remaining = (len(xyz_files) - i) / rate if rate > 0 else 0
            print(
                f'  [{i}/{len(xyz_files)}] '
                f'valid: {len(cached_samples)} | '
                f'failed: {n_failed_xyz + n_failed_parse + n_failed_rdkit} | '
                f'elapsed: {elapsed/60:.1f}min | '
                f'eta: {remaining/60:.1f}min'
            )

        # step 1: parse xyz file to get smiles and target values.
        try:
            smiles, targets_dict = parse_xyz_file(filepath)
        except Exception:
            n_failed_xyz += 1
            continue

        # step 2: encode smiles through the zinc cfg.
        # this is the slow step. each molecule takes ~0.01-0.05s.
        try:
            with _suppress_stderr():
                rule_indices = encode_smiles(smiles)
            tokens_idx, atom_mask = construct_token_sequence(rule_indices, max_len=500)
        except Exception:
            n_failed_parse += 1
            continue

        # step 3: build rdkit graph features for the gcn branch.
        try:
            result = smiles_to_graph_features(smiles)
            if result is None:
                n_failed_rdkit += 1
                continue
            x, edge_index, edge_attr = result
        except Exception:
            n_failed_rdkit += 1
            continue

        # step 4: assemble target vector [mu, alpha, homo, lumo, gap].
        targets_array = np.array([
            targets_dict['mu'],
            targets_dict['alpha'],
            targets_dict['homo'],
            targets_dict['lumo'],
            targets_dict['gap'],
        ], dtype=np.float32)

        cached_samples.append({
            'tokens_idx': tokens_idx,
            'atom_mask':  atom_mask,
            'targets':    targets_array,
            'x':          x,
            'edge_index': edge_index,
            'edge_attr':  edge_attr,
        })

    elapsed = time.time() - t_start
    print()
    print(f'preprocessing complete.')
    print(f'  valid molecules:  {len(cached_samples)}')
    print(f'  xyz parse failed: {n_failed_xyz}')
    print(f'  cfg parse failed: {n_failed_parse}')
    print(f'  rdkit failed:     {n_failed_rdkit}')
    print(f'  total time:       {elapsed/60:.1f} minutes')
    print()

    return cached_samples


# ============================================================
# section 8: dataset wrapper that uses the cache
# ============================================================

class CachedQM9Dataset(Dataset):
    # wraps the preprocessed molecule cache.
    # does not re-encode anything. only selects the target column.
    # to train on a different target, create a new instance of this class
    # with a different target_col value, pointing to the same cached_samples.

    def __init__(self, cached_samples, target_col):
        # cached_samples : list of dicts from preprocess_all_molecules()
        # target_col     : integer index into the 5-element targets array
        #                  0=mu, 1=alpha, 2=homo, 3=lumo, 4=gap
        self.samples    = cached_samples
        self.target_col = target_col

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            torch.from_numpy(s['tokens_idx']).long(),
            torch.from_numpy(s['atom_mask']).long(),
            torch.tensor([s['targets'][self.target_col]], dtype=torch.float),
            s['x'],
            s['edge_index'],
            s['edge_attr'],
        )


def qm9_collate_fn(batch):
    # collate a list of samples from CachedQM9Dataset into batch tensors.
    #
    # transformer inputs:
    #   tokens_idx_batch : [batch, 501] long
    #   atom_mask_batch  : [batch, 501] long
    #   target_batch     : [batch, 1]   float
    #
    # gcn inputs use global node re-indexing (pytorch geometric convention):
    #   each molecule's atom indices start from 0. when batching, they are
    #   offset by the cumulative node count so all molecules share one index space.

    batch_size = len(batch)

    tokens_idx_batch = torch.zeros(batch_size, 501, dtype=torch.long)
    atom_mask_batch  = torch.zeros(batch_size, 501, dtype=torch.long)
    target_batch     = torch.zeros(batch_size, 1,   dtype=torch.float)

    x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
    num_nodes = 0

    for i, (tokens_idx, atom_mask, target, x, edge_index, edge_attr) in enumerate(batch):
        tokens_idx_batch[i] = tokens_idx
        atom_mask_batch[i]  = atom_mask
        target_batch[i]     = target

        x_list.append(x)
        edge_index_list.append(edge_index + num_nodes)
        edge_attr_list.append(edge_attr)
        batch_list.append(torch.full((x.size(0),), i, dtype=torch.long))
        num_nodes += x.size(0)

    return (
        tokens_idx_batch,
        atom_mask_batch,
        target_batch,
        torch.cat(x_list,          dim=0),
        torch.cat(edge_index_list,  dim=1),
        torch.cat(edge_attr_list,   dim=0),
        torch.cat(batch_list,       dim=0),
    )


# ============================================================
# section 9: training and evaluation
# ============================================================

def train_one_epoch(model, loader, optimizer, device):
    # one full pass over the training set.
    # returns the average mse loss across all batches.

    model.train()
    total_loss = 0.0
    n_batches  = 0

    for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in loader:
        token_idx  = token_idx.to(device)
        target     = target.to(device)
        x          = x.to(device)
        edge_index = edge_index.to(device)
        edge_attr  = edge_attr.to(device)
        batch      = batch.to(device)

        optimizer.zero_grad()

        pred = model(token_idx, x, edge_index, edge_attr, batch).squeeze(-1)
        tgt  = target.squeeze(-1)

        if torch.isnan(pred).any():
            continue

        loss = F.mse_loss(pred, tgt)
        loss.backward()
        # gradient clipping prevents exploding gradients during early training.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / max(n_batches, 1)


def evaluate(model, loader, device):
    # inference pass. returns mean absolute error over all samples.

    model.eval()
    all_pred, all_true = [], []

    with torch.no_grad():
        for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in loader:
            token_idx  = token_idx.to(device)
            x          = x.to(device)
            edge_index = edge_index.to(device)
            edge_attr  = edge_attr.to(device)
            batch      = batch.to(device)
            target     = target.to(device)

            pred = model(token_idx, x, edge_index, edge_attr, batch)
            if torch.isnan(pred).any():
                continue

            all_pred.append(pred.squeeze(-1).cpu())
            all_true.append(target.squeeze(-1).cpu())

    if not all_pred:
        return float('nan')

    all_pred = torch.cat(all_pred)
    all_true = torch.cat(all_true)
    return torch.mean(torch.abs(all_pred - all_true)).item()


def build_model(device):
    # instantiate the full model and move to the target device.

    transformer = TransformerEncoder(
        d_model=D_MODEL, n_layers=N_LAYERS, vocab_size=VOCAB_SIZE,
        maxlen=MAXLEN, d_k=D_K, d_v=D_V, n_heads=N_HEADS, d_ff=D_FF,
        electron_token_set=ELECTRON_RELEVANT_TOKEN_SET,
    )
    gcn = GCN(
        num_layer=GCN_LAYERS, emb_dim=GCN_EMB_DIM,
        feat_dim=GCN_FEAT_DIM, drop_ratio=GCN_DROP,
    )
    combined  = CombinedModel(transformer, gcn)
    predictor = Predictor(in_dim=COMBINED_DIM, dropout=0.2)
    return FullModel(combined, predictor).to(device)


# ============================================================
# section 10: main benchmark loop
# ============================================================

def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'device          : {device}')
    print(f'quick_run       : {QUICK_RUN}')
    print(f'epochs / target : {EPOCHS}')
    print(f'batch size      : {BATCH_SIZE}')
    print(f'qm9 xyz dir     : {QM9_XYZ_DIR}')
    print()

    # step 1: preprocess all molecules once.
    # this is the most time-consuming step (~30-45 minutes on t4 for 130k molecules).
    # all five subsequent training loops share this cache.
    cached_samples = preprocess_all_molecules(QM9_XYZ_DIR, max_molecules=MAX_MOLECULES)

    if len(cached_samples) == 0:
        raise RuntimeError('no valid molecules were loaded. check QM9_XYZ_DIR and file format.')

    # compute the train/val/test split indices once.
    # the same split is reused for all five targets so results are comparable.
    n         = len(cached_samples)
    train_n   = int(n * TRAIN_RATIO)
    val_n     = int(n * VAL_RATIO)
    test_n    = n - train_n - val_n

    # generate indices using a seeded generator for reproducibility.
    rng     = torch.Generator().manual_seed(SEED)
    perm    = torch.randperm(n, generator=rng).tolist()
    train_indices = perm[:train_n]
    val_indices   = perm[train_n:train_n + val_n]
    test_indices  = perm[train_n + val_n:]

    print(f'dataset split: train={train_n}, val={val_n}, test={test_n}')
    print()

    results = {}

    # step 2: loop over the five targets.
    # for each target, we create a CachedQM9Dataset pointing to the same
    # cached_samples list but with a different target_col.
    # no re-parsing happens inside this loop.
    for target_info in TARGETS:
        target_name = target_info['name']
        target_col  = target_info['col']

        print(f'=== target {target_col + 1}/5: {target_name} ===')

        # build datasets using the same cached features, different label column.
        full_dataset = CachedQM9Dataset(cached_samples, target_col=target_col)

        train_set = torch.utils.data.Subset(full_dataset, train_indices)
        val_set   = torch.utils.data.Subset(full_dataset, val_indices)
        test_set  = torch.utils.data.Subset(full_dataset, test_indices)

        train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                                  collate_fn=qm9_collate_fn, num_workers=2,
                                  pin_memory=(device.type == 'cuda'))
        val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                                  collate_fn=qm9_collate_fn, num_workers=2,
                                  pin_memory=(device.type == 'cuda'))
        test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                                  collate_fn=qm9_collate_fn, num_workers=2,
                                  pin_memory=(device.type == 'cuda'))

        # fresh model for each target. each target is an independent experiment.
        model = build_model(device)

        optimizer = optim.Adam(
            model.parameters(), lr=LEARNING_RATE,
            betas=(0.9, 0.98), weight_decay=WEIGHT_DECAY,
        )
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=5, min_lr=1e-6,
        )

        best_val_mae      = float('inf')
        best_test_mae     = float('inf')
        best_model_state  = None

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()

            train_loss = train_one_epoch(model, train_loader, optimizer, device)
            val_mae    = evaluate(model, val_loader,  device)
            test_mae   = evaluate(model, test_loader, device)

            scheduler.step(val_mae)

            # read the current learned electron_bias value.
            # this is the key diagnostic: if the bias grows toward a larger value
            # during training, it means the model is using the electron-relevance
            # signal. if it shrinks toward 0, the signal was not useful for this target.
            bias_val = model.combined.transformer.embedding.electron_bias.item()

            print(
                f'  epoch {epoch:03d}/{EPOCHS} | '
                f'loss: {train_loss:.5f} | '
                f'val mae: {val_mae:.5f} | '
                f'test mae: {test_mae:.5f} | '
                f'lr: {optimizer.param_groups[0]["lr"]:.1e} | '
                f'e_bias: {bias_val:.5f} | '
                f't: {time.time()-t0:.1f}s'
            )

            if val_mae < best_val_mae:
                best_val_mae     = val_mae
                best_test_mae    = test_mae
                best_model_state = copy.deepcopy(model.state_dict())

        # reload best checkpoint and do final test evaluation.
        model.load_state_dict(best_model_state)
        final_test_mae = evaluate(model, test_loader, device)
        final_bias     = model.combined.transformer.embedding.electron_bias.item()

        print(f'  best val mae   : {best_val_mae:.5f}')
        print(f'  final test mae : {final_test_mae:.5f}')
        print(f'  learned e_bias : {final_bias:.6f}')
        print()

        results[target_name] = {
            'best_val_mae':   best_val_mae,
            'final_test_mae': final_test_mae,
            'learned_bias':   final_bias,
        }

    # step 3: print the final summary table.
    print('=' * 72)
    print('benchmark summary — property-aware molgramtreenet on qm9')
    print('=' * 72)
    print(f'  model variant  : electron-bias-v1 (option 1, init=0.1)')
    print(f'  electron rules : {sorted(ELECTRON_RELEVANT_TOKEN_SET)}')
    print(f'  epochs         : {EPOCHS} | batch: {BATCH_SIZE}')
    print(f'  d_model        : {D_MODEL} | layers: {N_LAYERS} | heads: {N_HEADS}')
    print()
    header = f'  {"target":<40} {"val mae":>10} {"test mae":>10} {"e_bias":>10}'
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for name, r in results.items():
        print(
            f'  {name:<40} '
            f'{r["best_val_mae"]:>10.5f} '
            f'{r["final_test_mae"]:>10.5f} '
            f'{r["learned_bias"]:>10.6f}'
        )
    print('=' * 72)

    # step 4: save results to csv.
    rows = [
        {
            'target':         name,
            'best_val_mae':   r['best_val_mae'],
            'final_test_mae': r['final_test_mae'],
            'learned_bias':   r['learned_bias'],
        }
        for name, r in results.items()
    ]
    pd.DataFrame(rows).to_csv('qm9_benchmark_results.csv', index=False)
    print('results saved to qm9_benchmark_results.csv')


if __name__ == '__main__':
    main()


installing dependencies... (using pre-built wheels to prevent hanging)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 51.5 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.6 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 55.9 MB/s eta 0:00:00


...

  epoch 045/50 | loss: 0.30665 | val mae: 0.54691 | test mae: 0.54700 | lr: 2.0e-04 | e_bias: -0.00011 | t: 855.4s


  epoch 046/50 | loss: 0.29896 | val mae: 0.55963 | test mae: 0.55921 | lr: 2.0e-04 | e_bias: 0.00017 | t: 854.9s


  epoch 047/50 | loss: 0.29265 | val mae: 0.54216 | test mae: 0.54942 | lr: 2.0e-04 | e_bias: -0.00020 | t: 855.6s


  epoch 048/50 | loss: 0.28643 | val mae: 0.54954 | test mae: 0.54985 | lr: 1.4e-04 | e_bias: 0.00033 | t: 855.0s


KeyboardInterrupt: 